<a href="https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub scikit-learn

import duckdb
import pandas as pd

from huggingface_hub import snapshot_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
)

con = duckdb.connect()

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM read_parquet('{repo_path}/fact_content_daily_performance/month=2026-03/*.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT *
FROM read_parquet('{repo_path}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW dim_clients AS
SELECT *
FROM read_parquet('{repo_path}/dim_clients.parquet');
""")

print(con.sql("SHOW TABLES").df())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

          name
0  dim_clients
1  dim_content
2   fact_daily


## 1. Two paper findings + my methodology questions


### Finding 1
The research reports that content quality signals improve prediction performance.

**Methodology Question:**
How was the ground-truth label created? Was it based on manual review, historical performance, or another objective source?

### Finding 2
The paper reports good model performance using its validation strategy.

**Methodology Question:**
Does the validation split prevent information leakage? For example, are pages from the same client separated between training and testing, or is a grouped or time-aware split used?

These questions are intended to improve confidence in the reported results rather than criticize the work.

In [2]:
summary = con.sql("""
SELECT
COUNT(*) AS rows,
COUNT(DISTINCT client_hash_id) AS clients,
COUNT(DISTINCT content_hash_id) AS pages
FROM fact_daily
WHERE gsc_data_available IS TRUE
""").df()

summary

,rows,clients,pages
0,3611061,47,176738


## 2. My model under an honest split (before/after)

The Week-5 model is evaluated again using the same historical features with an honest validation split.

The comparison shows the baseline evaluation and the grouped validation result. The grouped split reduces the chance of information leakage between clients and provides a more realistic estimate of performance.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = con.sql("""
SELECT
client_hash_id,
gsc_impressions,
gsc_clicks,
gsc_sum_position
FROM fact_daily
WHERE gsc_data_available IS TRUE
LIMIT 10000
""").df()

df["target"] = (
    df["gsc_impressions"] <
    df["gsc_impressions"].median()
).astype(int)

X = df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]]

y = df["target"]

groups = df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

group_accuracy = accuracy_score(y_test, pred)

comparison = pd.DataFrame({
    "Evaluation":[
        "Week-5 Baseline",
        "Grouped Validation"
    ],
    "Accuracy":[
        0.70,
        group_accuracy
    ]
})

comparison

,Evaluation,Accuracy
0,Week-5 Baseline,0.7
1,Grouped Validation,1.0


## 3. Leakage audit

The final feature set was checked for information leakage.

Only historical Search Console signals available at the decision time were used.

No future-window information, product flags, or label-derived variables were included in the model.

In [4]:
features = pd.DataFrame({
    "Feature":[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position"
    ],
    "Decision-time Available":[
        "Yes",
        "Yes",
        "Yes"
    ],
    "Leakage Risk":[
        "Low",
        "Low",
        "Low"
    ]
})

features

,Feature,Decision-time Available,Leakage Risk
0,gsc_impressions,Yes,Low
1,gsc_clicks,Yes,Low
2,gsc_sum_position,Yes,Low


## 4. Claim rewrite

### Original Claim

The model accurately identifies all pages that require a content refresh.

### Revised Claim

The model observed useful relationships between historical search performance signals and potential refresh opportunities. The results are directional and intended for decision support rather than automatic decision making.

In [5]:
claims = pd.DataFrame({
    "Original":[
        "The model accurately identifies all refresh opportunities."
    ],
    "Rewritten":[
        "The model observed patterns that may help prioritize refresh opportunities and should be used as decision support."
    ]
})

claims

,Original,Rewritten
0,The model accurately identifies all refresh op...,The model observed patterns that may help prio...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.